In [18]:
from pathlib import Path
import re
import os

input_dir = Path("/mnt/nfs_dev/zah/data/book/JD_V2")
files = sorted([f for f in input_dir.rglob("*.md") if 'debug' not in f.parts])
print(files)

[PosixPath('/mnt/nfs_dev/zah/data/book/JD_V2/150种人造板材配方与制作 (李东光主编).pdf/150种人造板材配方与制作 (李东光主编).pdf.md'), PosixPath('/mnt/nfs_dev/zah/data/book/JD_V2/2011年西门子自动化专家会议论文集 上 (西门子（中国）有限公司编) .pdf/2011年西门子自动化专家会议论文集 上 (西门子（中国）有限公司编) .pdf.md'), PosixPath('/mnt/nfs_dev/zah/data/book/JD_V2/2011年西门子自动化专家会议论文集 下 (西门子（中国）有限公司编) .pdf/2011年西门子自动化专家会议论文集 下 (西门子（中国）有限公司编) .pdf.md'), PosixPath('/mnt/nfs_dev/zah/data/book/JD_V2/3D Integration for VLSI Systems (Chuan Seng Tan).pdf/3D Integration for VLSI Systems (Chuan Seng Tan).pdf.md'), PosixPath('/mnt/nfs_dev/zah/data/book/JD_V2/802.11无线网络权威指南 (Matthew S. Gast) .pdf/802.11无线网络权威指南 (Matthew S. Gast) .pdf.md'), PosixPath('/mnt/nfs_dev/zah/data/book/JD_V2/802.11无线网络权威指南 (Matthew S. Gast) .pdf/802.11无线网络权威指南 (Matthew S. Gast) .pdf_2.md'), PosixPath('/mnt/nfs_dev/zah/data/book/JD_V2/ADINA有限元经典实例分析 (马野，袁志丹，曹金风编著, 马野, author) .pdf/ADINA有限元经典实例分析 (马野，袁志丹，曹金风编著, 马野, author) .pdf.md'), PosixPath('/mnt/nfs_dev/zah/data/book/JD_V2/ADINA有限元经典实例分析 (马野，袁志丹，曹金风编著, 马野, a

In [19]:
def has_chinese_char(text):
    """
    检测字符串中是否包含中文字符
    原理：检查字符的 Unicode 编码是否在常用汉字范围内
    """
    for char in text:
        if '\u4e00' <= char <= '\u9fff':
            return True
    return False

def is_english_file(filename):
    """
    判断文件是否为英文文件
    规则：文件名中不存在任何中文字符，即为英文文件
    """
    # 1. 获取文件名主体（去除后缀）
    # 兼容处理：如果输入是字符串路径，先转为 Path 对象
    if isinstance(filename, str):
        p = Path(filename)
    else:
        p = filename
    
    name_without_ext = p.stem
    
    # 2. 核心逻辑：如果没有中文字符，就认为是英文文件
    # 如果检测函数返回 False (无中文)，则 is_english 为 True
    return not has_chinese_char(name_without_ext)

In [20]:
def extract_entoc_robust(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    start_idx = -1
    for i, line in enumerate(lines):
        clean_line = line.strip().replace(" ", "")
        if re.search(r'^#*CONTENTS$', clean_line, re.I) or re.search(r'^#*Contents at a glance$', clean_line, re.I):
            start_idx = i
            break
            
    if start_idx == -1: return "False", None, None, file_path

    # --- 阶段 1: 采样前 5-8 行有效标题作为锚点集合 ---
    anchor_set = []
    anchor_clean_set = []
    scan_limit = 8 # 稍微多扫几行以确保抓到核心标题
    found_count = 0
    
    for i in range(start_idx + 1, int(len(lines)*0.2)):
        line_content = lines[i].strip().replace("#", "").strip()
        if not line_content: continue

        pure_title = re.sub(r'\s+\d+$', '', line_content).strip()
        anchor_clean_set.append(pure_title.replace(" ", "") )
        
        # if pure_title in anchor_set or pure_title in anchor_clean_set:
        #     return anchor_set
        
        anchor_set.append(pure_title)
        found_count += 1
            
        if found_count >= scan_limit or i > start_idx + 20: # 找到5个或扫描超过20行就停止
            break
    # anchor_clean_set =  [item.replace(" ", "") for item in anchor_set]
    end_start_idx = i + 1

    # --- 阶段 2: 寻找终点 ---
    end_idx = -1
    i = end_start_idx
    limit = int(len(lines) * 0.2)
    
    while i < limit:
        line_raw = lines[i].strip()
        clean_content = line_raw.replace("#", "").strip().replace(" ", "")
        
        # 1. 检查是否匹配锚点集合 (正文标题重现)
        is_anchor_match = clean_content and any(clean_content.lower() in anchor.lower() for anchor in anchor_set)
        is_anchor_clean_match = clean_content and any(clean_content.lower() in anchor.lower() for anchor in anchor_clean_set)
        is_match = any(clean_content == anchor.replace(" ", "") for anchor in anchor_set)
        has_page_num = re.search(r'\s\d+$', line_raw)
    
        if (is_anchor_match and not has_page_num) or(is_anchor_clean_match and not has_page_num) or  is_match:
            end_idx = i
            break
    
        # 2. 兜底逻辑：贪婪匹配参考文献
        # 匹配规则：去掉符号、数字、标点后，内容等于“参考文献”等
        # 使用 re.sub 去掉非中文字符和非英文字母
        simplified_content = re.sub(r'[^\u4e00-\u9fa5a-zA-Z]', '', clean_content)
        is_ref_header = re.match(r'^(Index|Bibliography|References)$', simplified_content, re.IGNORECASE)
    
        if is_ref_header:
            # 初始标记为当前行
            end_idx = i + 1
            
            # 开启贪婪探测模式：向后查找 50 行
            look_ahead_idx = i + 1
            while look_ahead_idx < min(i + 51, limit):
                next_line = lines[look_ahead_idx].strip()
                next_clean = next_line.replace("#", "").strip().replace(" ", "")
                next_simplified = re.sub(r'[^\u4e00-\u9fa5a-zA-Z]', '', next_clean)
                
                # 如果在 50 行内又发现了一个独立的“参考文献”行
                if re.match(r'^(Index|Bibliography|References)$', next_simplified, re.IGNORECASE):
                    end_idx = look_ahead_idx + 1
                    # 重置 i 到新发现的位置，以便外层循环能继续从这里开始下一次 50 行的探测
                    i = look_ahead_idx 
                    # 更新 look_ahead_idx 重新开始算 50 行
                    look_ahead_idx = i + 1
                    continue
                
                look_ahead_idx += 1
            
            # 探测结束，跳出外层循环
            break
        
        i += 1

    # --- 阶段 3: 返回截取结果 ---
    if end_idx != -1:
        # 如果是分行标题，end_idx 可能是标题的第二行，视情况可以向上调1行
        return lines[start_idx:end_idx], start_idx, end_idx, file_path
    else:
        # 极端情况兜底：取目录开始后的300行
        return "False", None, None, file_path
    

In [21]:
false_md = []
toc_list = []
for i in range(len(files)):
    if is_english_file(files[i]):
        toc_blocks, start_idx, end_idx, file_path = extract_entoc_robust(files[i])
        if toc_blocks == "False":
            false_md.append(files[i])
        else:
            if toc_blocks == "False":
                false_md.append(files[i]) 
            else:
                toc_blocks = [block for block in toc_blocks if len(block) <= 200]
                print(f"------------------------{i}--------------------------")
                print(toc_blocks)
                toc_list.append((toc_blocks, start_idx, end_idx, file_path))
    else:
        pass
print(false_md)


------------------------3--------------------------
['# Contents\n', '\n', 'Preface v\n', '\n', 'Contents vii\n', '\n', 'Chapter 1 3D Integration Technology - Introduction 1\n', '\n', 'and Overview\n', '\n', 'Chuan Seng Tan, Kuan-Neng Chen and\n', '\n', 'Steven J. Koester\n', '\n', 'Chapter 2 A Systems Perspective on 3D Integration: 27\n', '\n', 'What is 3D? And What is 3D Good For?\n', '\n', 'Phil Emma and Eren Kursun\n', '\n', 'Chapter 3 Wafer Bonding Techniques 43\n', '\n', 'Bioh Kim, Thorsten Matthias, Viorel Dragoi,\n', '\n', 'Markus Wimplinger and Paul Lindner\n', '\n', 'Chapter 4 TSV Etching 71\n', '\n', 'Paul Werbaneth\n', '\n', 'Chapter 5 TSV Filling 91\n', '\n', 'Arthur Keigler\n', '\n', 'Chapter 6 3D Technology Platform: Temporary Bonding 121\n', '\n', 'and Release\n', '\n', 'Mark Privett\n', '\n', 'Chapter 7 3D Technology Platform: Wafer Thinning, 139\n', '\n', 'Stress Relief, and Thin Wafer Handling\n', '\n', 'Scott Sullivan\n', '\n', 'Chapter 8 Advanced Die-to-Wafer 3D In

In [22]:
def auto_detect_hierarchy(md_content):
    # 1. 预定义可能的特征提取正则 (按优先级排序)
    feature_patterns = [
        r'(?i)part\s+\d+',
        r'(?i)part\s+[ⅰⅱⅲⅳⅴⅵⅶⅷⅸⅹ]+',
        r'(?i)chapter\s+\d+',
        r'(?i)section\s+\d+',
        r'\d+\-\d+\-\d+',  # 1-1-1
        r'\d+\-\d+',      # 1-1
        r'\d+\.\d+\.\d+', # 1.1.1
        r'\d+\.\d+',      # 1.1
        r'\d+[\.．]\s*',   # 1. (带点)
        r'^\d+\s',        # 1 (空格)
        r'^\(\d+\)',      # (1)
    ]

    level_queue = []
    output_lines = []
    
    for line in md_content:
        raw_content = line.strip()
        raw_content = raw_content.lstrip('#')
        raw_content = raw_content.strip()
        if not raw_content: continue
        
        # 移除行首旧的 # 和行尾页码
        clean_text = re.sub(r'^#+\s*', '', raw_content)
        clean_text = re.sub(r'[\.\…]{2,}', '', clean_text)
        pattern = r'\s*\(\d+\)\s*$|\s+\d+$'
        clean_text = re.sub(pattern, '', clean_text, flags=re.MULTILINE)
        
        current_fingerprint = None
        
        # 匹配特征
        for p in feature_patterns:
            match = re.search(p, clean_text)
            if match:
                normalized_match = re.sub(r'[\.、]\s*', '. ', match.group())
                
                current_fingerprint = re.sub(r'\d+', '\\\\d', normalized_match)
                current_fingerprint = re.sub(r'[ⅰⅱⅲⅳⅴⅵⅶⅷⅸⅹ]+', 'ROMAN', current_fingerprint, flags=re.IGNORECASE)
                break
        
        if current_fingerprint:
            if current_fingerprint not in level_queue:
                level_queue.append(current_fingerprint)
            
            depth = level_queue.index(current_fingerprint) + 1
            output_lines.append(f"{'#' * depth} {clean_text}")
        else:
            if raw_content.startswith("#"):
                output_lines.append(raw_content)
            
    return output_lines

In [23]:
processed_toc_list = []
for idx, toc in enumerate(toc_list):
    toc_content, start_idx, end_idx, file_path = toc
    processed_toc = auto_detect_hierarchy(toc_content)
    processed_toc_list.append((processed_toc, start_idx, end_idx, file_path))
# for idx, item in enumerate(processed_toc_list):
#     processwd_toc, start_idx, end_idx, file_path = item
#     rewrite_md(processwd_toc, start_idx, end_idx, file_path)

In [24]:
i = 0
processed_toc, start_idx, end_idx, file_path = processed_toc_list[i]

In [25]:
file_path

PosixPath('/mnt/nfs_dev/zah/data/book/JD_V2/3D Integration for VLSI Systems (Chuan Seng Tan).pdf/3D Integration for VLSI Systems (Chuan Seng Tan).pdf.md')

In [26]:
processed_toc

['# Chapter 1 3D Integration Technology - Introduction',
 '# Chapter 2 A Systems Perspective on 3D Integration:',
 '# Chapter 3 Wafer Bonding Techniques',
 '# Chapter 4 TSV Etching',
 '# Chapter 5 TSV Filling',
 '# Chapter 6 3D Technology Platform: Temporary Bonding',
 '# Chapter 7 3D Technology Platform: Wafer Thinning,',
 '# Chapter 8 Advanced Die-to-Wafer 3D Integration',
 '# Chapter 9 Advanced Direct Bond Technology',
 '# Chapter 10 Surface Modification Bonding at Low',
 '# Chapter 11 Through Silicon Via Implementation in CMOS 231 Image Sensor Product Xavier Gagnard and Nicolas Hotellier',
 '# Chapter 12 A 300-mm Wafer-Level Three-Dimensional Integration Scheme Using Tungsten Through-Silicon Via and Hybrid Cu-Adhesive Bonding Fei Liu',
 '# Chapter 13 Power Delivery in 3D IC Technology with a Stratum Having an Array of Monolithic DC-DC Point-of-Load (PoL) Converter Cells Ron Rutman and Jian Sun',
 '# Chapter 14 Thermal-Aware 3D IC Designs 313 Xiaoxia Wu, Yuan Xie and Vijaykirshnan N

In [27]:
with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

In [28]:
type(lines)
print(start_idx, end_idx)

108 188


In [29]:
lines = lines[end_idx:]
lines

['\n',
 '# 3D INTEGRATION TECHNOLOGY - INTRODUCTION AND OVERVIEW\n',
 '\n',
 'Chuan Seng Tan\n',
 '\n',
 'Nanyang Technological University\n',
 '\n',
 'Steven J. Koester\n',
 '\n',
 'University of Minnesota\n',
 '\n',
 'Kuan-Neng Chen\n',
 '\n',
 'National Chiao Tung University\n',
 '\n',
 '# 1.1 INTRODUCTION\n',
 '\n',
 'The past decade has seen three-dimensional (3D) integration technology mature rapidly from a hypothetical concept to a technology that is on the cusp widespread commercial implementation. This rapid trend towards acceptance of 3D integration has been both a result of key demonstrations of the technical feasibility of the process, as well as a growing consensus that 3D integration will be necessary to continue current computational system performance trends. 3D technology also offers an abundance of opportunities for new applications and functionality. In this introduction, we provide an overview of the system needs that are driving 3D integration development, the rece

In [ ]:
processed_toc

In [ ]:
type(processed_toc)

In [ ]:
import re

def sync_toc_to_markdown(input_path, processed_toc, start_idx, end_idx):
    with open(input_path, 'r', encoding='utf-8') as f:
        original_lines = f.readlines()
        lines = original_lines[end_idx:]
    # 预处理 TOC：保留 (清洗后的文字, 原始标准格式) 的元组列表
    # 这样我们可以通过索引顺序访问
    clean_toc = []
    for t in processed_toc:
        clean_key = re.sub(r'[#\s]', '', t)
        clean_toc.append({'key': clean_key, 'full': t})

    new_lines = original_lines[:end_idx]
    toc_ptr = 0  # 当前正在寻找的 TOC 标题索引
    line_idx = 0
    num_lines = len(lines)
    num_toc = len(clean_toc)

    while line_idx < num_lines:
        line = lines[line_idx]
        stripped_line = line.strip()

        # 跳过空行，原样保留
        if not stripped_line:
            new_lines.append(line)
            line_idx += 1
            continue

        found_match = False
        
        # 核心逻辑：从当前 toc_ptr 开始往后找标题
        # 通常标题就在当前指针，但也可能文中缺失了某个标题，所以往后多看几个
        # 限制只往后看 3 个，防止跨度过大导致误匹配
        for look_ahead_toc in range(toc_ptr, min(toc_ptr + 3, num_toc)):
            target_key = clean_toc[look_ahead_toc]['key']
            
            # 滑动窗口尝试合并多行匹配当前 target_key
            temp_text = ""
            for window_size in range(0, 5): # 向后合并最多 5 行非空行
                if line_idx >= num_lines:
                    break
                if len(lines[line_idx]) > 100:

                    line_idx += 1
                if line_idx + window_size >= num_lines:
                    break
                
                curr_content = lines[line_idx + window_size].strip()
                if not curr_content and window_size > 0:
                    continue # 遇到中间空行，继续合并下一行内容
                
                temp_text += re.sub(r'[#\s]', '', curr_content)
                
                if temp_text == target_key:
                    # 匹配成功！
                    new_lines.append(clean_toc[look_ahead_toc]['full'] + '\n')
                    # 更新 TOC 指针：下次从这个标题的下一个开始找
                    toc_ptr = look_ahead_toc + 1
                    # 更新 Line 指针：跳过参与合并的所有行
                    line_idx += window_size + 1
                    found_match = True
                    break
            
            if found_match:
                break
        
        if found_match:
            continue

        # --- 如果没有匹配到 TOC ---
        if stripped_line.startswith('#'):
            # 情况：不在 TOC 中但是以 # 开头 -> 去掉 #
            new_lines.append(line.lstrip('#').lstrip())
        else:
            # 情况：普通正文 -> 原样保留
            new_lines.append(line)
        
        line_idx += 1

    return new_lines

def process_file(input_path, processed_toc, start_idx, end_idx):

    # 2. 执行转换逻辑
    result_lines = sync_toc_to_markdown(input_path, processed_toc, start_idx, end_idx)

    # 3. 构造输出路径：在原文件名后加 _2
    # splitext 分离文件名和后缀，例如 ('test', '.md')
    base_path, ext = os.path.splitext(input_path)
    output_path = f"{base_path}_2{ext}"

    # 4. 保存文件
    with open(output_path, 'w', encoding='utf-8') as f:
        f.writelines(result_lines)
    
    print(f"处理完成！\n源文件: {input_path}\n新文件: {output_path}")


# process_file(file_path, processed_toc, lines)

# 打印结果查看
# print("".join(result))

In [ ]:
for idx, item in enumerate(processed_toc_list):
    processwd_toc, start_idx, end_idx, file_path = item
    process_file(file_path, processwd_toc, start_idx, end_idx)